In [ ]:
import pandas as pd
import numpy as np

# Nacional

In [ ]:
year = "2017"
data = pd.read_csv(f'df_ecv_{year}.csv')
print(data['DIRECTORIO'].nunique())
print(len(data))

8501
8612


In [ ]:
data[data.duplicated(keep=False)]

,DIRECTORIO,YEAR,CLASE,P4005,P4015,P8520S1,P8520S3,P8520S4,P8520S5,CANT_HOGARES_VIVIENDA,CANT_PERSONAS_HOGAR,P5010,P8526,P8530,P764,FEX_C_2018
224,6000311,2017,1,1,4,1,1,1,1,2,3,1,1,1,1,767.578517
225,6000311,2017,1,1,4,1,1,1,1,2,3,1,1,1,1,767.578517
269,6000366,2017,1,1,6,1,1,1,1,2,1,1,1,1,1,363.095963
270,6000366,2017,1,1,6,1,1,1,1,2,1,1,1,1,1,363.095963
318,6000416,2017,1,1,4,1,1,1,1,3,3,2,1,1,1,648.732794
319,6000416,2017,1,1,4,1,1,1,1,3,3,2,1,1,1,648.732794
438,6000604,2017,1,1,4,1,1,1,1,2,1,1,1,1,1,940.856058
439,6000604,2017,1,1,4,1,1,1,1,2,1,1,1,1,1,940.856058
442,6000607,2017,1,1,4,1,1,1,1,2,1,1,1,1,1,1171.972100
443,6000607,2017,1,1,4,1,1,1,1,2,1,1,1,1,1,1171.972100


In [ ]:
data['CLASIFICACION'] = 'Sin deficit'

In [ ]:
# Los materiales de las paredes exteriores son:
# "5 Madera burda, tabla, tablón
# 7 Guadua, caña, esterilla, otro vegetal
# 8 Zinc, tela, carbón, latas, desechos, plástico
# 9 Sin paredes"
condicion_deficit_cuantitativo = data["P4005"].isin([5, 7, 8, 9])
data['IS_MAT_PAREDES'] = condicion_deficit_cuantitativo.astype(int)

# 3 o más hogares por vivienda.
# Cabeceras y centros poblados: 2 o más hogares en la misma vivienda o 6 personas en total
# en la vivienda.
if "CANT_HOG_COMPLETOS" in data.columns:
    condicion_cohabitacion = (
        (
            (data["CANT_HOG_COMPLETOS"] >= 3)
            & (data["CLASE"] == 1)
        )
        |
        (
            (data["CANT_HOG_COMPLETOS"] >= 2)
            & (data["CLASE"].isin([2, 3]))
        )
    )

    condicion_deficit_cuantitativo |= condicion_cohabitacion
    data['IS_COHAB'] = condicion_cohabitacion.astype(int)

if "CANT_HOGARES_VIVIENDA" in data.columns:
    condicion_cohabitacion = (
        (
            (data["CANT_HOGARES_VIVIENDA"] >= 3)
            & (data["CLASE"] == 1)
        )
        |
        (
            (data["CANT_HOGARES_VIVIENDA"] >= 2)
            & (data["CLASE"].isin([2, 3]))
        )
    )

    condicion_deficit_cuantitativo |= condicion_cohabitacion
    data['IS_COHAB'] = condicion_cohabitacion.astype(int)

if "Cant_hog_completos" in data.columns:
    condicion_cohabitacion = (
        (
            (data["Cant_hog_completos"] >= 3)
            & (data["CLASE"] == 1)
        )
        |
        (
            (data["Cant_hog_completos"] >= 2)
            & (data["CLASE"].isin([2, 3]))
        )
    )

    condicion_deficit_cuantitativo |= condicion_cohabitacion
    data['IS_COHAB'] = condicion_cohabitacion.astype(int)

if "P70" in data.columns:
    condicion_cohabitacion = (
        (
            (data["P70"] >= 3)
            & (data["CLASE"] == 1)
        )
        |
        (
            (data["P70"] >= 2)
            & (data["CLASE"].isin([2, 3]))
        )
    )

    condicion_deficit_cuantitativo |= condicion_cohabitacion
    data['IS_COHAB'] = condicion_cohabitacion.astype(int)

# Aplica solo para las cabeceras municipales y sus centros poblados = hogares con más de
# cuatro personas por cuarto para dormir.
if "P205" in data.columns:
    condicion_hacinamiento_mitigable = (
        (data["P5010"].ne(0))
        & ((data["P205"] / data["P5010"]) > 4)
        & (data["CLASE"].isin([1, 2]))
    )
    condicion_deficit_cuantitativo |= condicion_hacinamiento_mitigable
    data['IS_H_MITIG'] = condicion_cohabitacion.astype(int)

if "Cant_personas_hogar" in data.columns:
    condicion_hacinamiento_mitigable = (
        (data["P5010"].ne(0))
        & ((data["Cant_personas_hogar"] / data["P5010"]) > 4)
        & (data["CLASE"].isin([1, 2]))
    )
    condicion_deficit_cuantitativo = condicion_hacinamiento_mitigable
    data['IS_H_MITIG'] = condicion_cohabitacion.astype(int)

if "CANT_PERSONAS_HOGAR" in data.columns:
    condicion_hacinamiento_mitigable = (
        (data["P5010"].ne(0))
        & ((data["CANT_PERSONAS_HOGAR"] / data["P5010"]) > 4)
        & (data["CLASE"].isin([1, 2]))
    )
    condicion_deficit_cuantitativo |= condicion_hacinamiento_mitigable
    data['IS_H_MITIG'] = condicion_cohabitacion.astype(int)

# Tipo de vivienda: Otro
if "P4000" in data.columns:
    condicion_tipo_vivienda = (data["P4000"] == 5)
    condicion_deficit_cuantitativo |= condicion_tipo_vivienda
    data['IS_MAT_PAREDES'] = condicion_tipo_vivienda.astype(int)

if "P1070" in data.columns:
    condicion_tipo_vivienda = (data["P1070"] == 5)
    condicion_deficit_cuantitativo |= condicion_tipo_vivienda
    data['IS_MAT_PAREDES'] = condicion_tipo_vivienda.astype(int)


alguna_cumple = condicion_deficit_cuantitativo.groupby(data["DIRECTORIO"]).transform("any")
primera_que_cumple = (
    condicion_deficit_cuantitativo
    & ~data.loc[condicion_deficit_cuantitativo, "DIRECTORIO"]
        .duplicated()
        .reindex(data.index, fill_value=False)
)


# 4. Conservar:
#    - todas las filas cuando ninguna cumple;
#    - solamente la primera que cumple cuando alguna cumple
data = data.loc[
    ~alguna_cumple | primera_que_cumple
].copy()

mask = (
    data["CLASIFICACION"].eq("Sin deficit")
    & condicion_deficit_cuantitativo
)

data.loc[mask, "CLASIFICACION"] = "Deficit cuantitativo"


In [ ]:
data

,DIRECTORIO,YEAR,CLASE,P4005,P4015,P8520S1,P8520S3,P8520S4,P8520S5,CANT_HOGARES_VIVIENDA,CANT_PERSONAS_HOGAR,P5010,P8526,P8530,P764,FEX_C_2018,CLASIFICACION,IS_MAT_PAREDES,IS_COHAB,IS_H_MITIG
0,6000000,2017,1,1,6,1,1,1,1,1,6,4,1,1,1,2046.919302,Sin deficit,0,0,0
1,6000001,2017,1,1,4,1,1,1,1,1,3,1,1,1,1,2337.288288,Sin deficit,0,0,0
2,6000002,2017,1,1,6,1,1,1,1,1,4,1,1,1,3,2512.507747,Sin deficit,0,0,0
3,6000003,2017,1,1,6,1,1,1,1,1,7,2,1,1,1,2221.313543,Sin deficit,0,0,0
4,6000005,2017,1,1,4,1,1,1,1,1,4,3,1,1,3,2125.036663,Sin deficit,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8607,6015980,2017,1,1,6,1,2,2,2,1,8,3,2,5,1,435.435666,Sin deficit,0,0,0
8608,6015981,2017,1,1,6,1,2,2,2,1,5,3,3,5,1,626.861723,Sin deficit,0,0,0
8609,6015982,2017,1,1,6,1,2,2,2,1,3,2,2,5,1,485.632845,Sin deficit,0,0,0
8610,6015983,2017,1,1,4,1,1,1,1,1,4,2,1,1,1,1607.345338,Sin deficit,0,0,0


In [ ]:

condicion_deficit_cualitativo = (
    # 3. Material predominante pisos
    # g. Tierra, arena
    data["P4015"].eq(7)
    |
    #Cabeceras municipales: hogares que habitan en viviendas sin conexión a acueducto.
    (
        data["P8520S5"].eq(2)
        & data["CLASE"].eq(1)
    )
    |
    (
        data["P8530"].isin([4, 5, 6, 9, 10])
        & data["CLASE"].isin([2, 3])
    )
    |
    # Energía: Hogares que habitan en viviendas sin servicio de energía eléctrica.
    data["P8520S1"].eq(2)
    |
    # Hogares que no cuentan con servicio de recolección de basuras.
    (
        data["P8520S4"].eq(2)
        & data["CLASE"].eq(1)
    )
    |
    #     Cabeceras municipales: hogares que habitan en viviendas:
    # Sin alcantarillado, o
    # Con alcantarillado, pero con servicio de sanitario conectado a pozo séptico o sin
    # conexión, letrina;
    # O con descarga directa a fuentes de agua (bajamar); o
    # Sin servicio de sanitario.
    (
        data["P8520S3"].eq(2)
        & data["CLASE"].eq(1)
    )
)

# Hacinamiento mitigable:
# Cabeceras municipales y centros poblados: más de cuatro personas por cuarto para dormir.
# Rural disperso: más de dos personas por cuarto para dormir.
if "P205" in data.columns:
    condicion_deficit_cualitativo |= (
        data["P5010"].ne(0)
        & ((data["P205"] / data["P5010"]) > 2)
        & data["CLASE"].eq(3)
    )

if "Cant_personas_hogar" in data.columns:
    condicion_deficit_cualitativo |= (
        data["P5010"].ne(0)
        & ((data["Cant_personas_hogar"] / data["P5010"]) > 2)
        & data["CLASE"].eq(3)
    )

if "CANT_PERSONAS_HOGAR" in data.columns:
    condicion_deficit_cualitativo |= (
        data["P5010"].ne(0)
        & ((data["CANT_PERSONAS_HOGAR"] / data["P5010"]) > 2)
        & data["CLASE"].eq(3)
    )

# ¿En dónde preparan los alimentos las personas de este hogar?
# "b. En un cuarto usado también para dormir
# d. En un comedor sin lavaplatos
# e. En un patio, corredor, enramada, al aire libre"
if "P8532" in data.columns:
    condicion_deficit_cualitativo |= ((
        data["P8532"].isin([2, 4, 5])
        & data["CLASE"].eq(1)
    )
    |
    (
        data["P8532"].isin([2, 4])
        & data["CLASE"].isin([2, 3])
    ))

if "P764" in data.columns:
    condicion_deficit_cualitativo |= ((
        data["P764"].isin([2, 4, 5])
        & data["CLASE"].eq(1)
    )
    |
    (
        data["P764"].isin([2, 4])
        & data["CLASE"].isin([2, 3])
    ))

# Tipo de Servicio Sanitario
    #     "b. Inodoro conectado a pozo séptico
    # c. Inodoro sin conexión
    # d. Letrina
    # e. Bajamar
    # f. No tiene servicio sanitario"
if "P8526" in data.columns:
    condicion_deficit_cualitativo |=((
        data["P8526"].isin([2, 3, 4, 5, 6])
        & data["CLASE"].isin([2, 3])
    ))

if "P8525" in data.columns:
    condicion_deficit_cualitativo |=((
        data["P8525"].isin([2, 3, 4, 5, 6])
        & data["CLASE"].isin([2, 3])
    ))

alguna_cumple = condicion_deficit_cualitativo.groupby(data["DIRECTORIO"]).transform("any")
primera_que_cumple = (
    condicion_deficit_cualitativo
    & ~data.loc[condicion_deficit_cualitativo, "DIRECTORIO"]
        .duplicated()
        .reindex(data.index, fill_value=False)
)


# 4. Conservar:
#    - todas las filas cuando ninguna cumple;
#    - solamente la primera que cumple cuando alguna cumple
data = data.loc[
    ~alguna_cumple | primera_que_cumple
].copy()

mask = (
    data["CLASIFICACION"].eq("Sin deficit")
    & condicion_deficit_cualitativo
)

data.loc[mask, "CLASIFICACION"] = "Deficit cualitativo"

In [ ]:
data[data['DIRECTORIO'].duplicated(keep=False)]['CLASIFICACION'].unique()

array(['Sin deficit'], dtype=object)

In [ ]:
data.to_csv(f'df_clasificado_{year}.csv', index=False)

In [ ]:
fex = data['FEX_C_2018']
data_agg = data.copy()

if "P4000" in data_agg.columns:
    data_agg = data_agg[data_agg["P4000"] != 4]
if "P1070" in data_agg.columns:
    data_agg = data_agg[data_agg["P1070"] != 4]

data_agg['total'] = fex
data_agg['sin_deficit'] = (data_agg['CLASIFICACION'] == 'Sin deficit') * fex
data_agg['deficit_cuanti'] = (data_agg['CLASIFICACION'] == 'Deficit cuantitativo') * fex
data_agg['deficit_cuali'] = (data_agg['CLASIFICACION'] == 'Deficit cualitativo') * fex
data_agg['tipo_viv'] = (data_agg['IS_MAT_PAREDES'] == 1) * fex
data_agg['mat_paredes'] = (data_agg['IS_MAT_PAREDES'] == 1) * fex
data_agg['cohab'] = (data_agg['IS_COHAB'] == 1) * fex
data_agg['h_mitigable'] = (data_agg['IS_H_MITIG'] == 1) * fex

columnas_a_agrupar = [
    'total', 'sin_deficit', 'deficit_cuanti', 'deficit_cuali', 'tipo_viv', 'mat_paredes', 'cohab', 'h_mitigable'
]

mapeo_clase = {
    1: 'Cabecera',
    2: 'Centros Poblados y Rural Disperso',
    3: 'Centros Poblados y Rural Disperso'
}
data_agg['CLASE_AGRUPADA'] = data_agg['CLASE'].map(mapeo_clase)

resultado_detallado = data_agg.groupby(['CLASE_AGRUPADA'])[columnas_a_agrupar].sum().reset_index()

resultado_total = data_agg[columnas_a_agrupar].sum().to_frame().T
resultado_total['CLASE_AGRUPADA'] = 'Total'

resultado_final = pd.concat([resultado_detallado, resultado_total], ignore_index=True)

orden_categorias = ['Cabecera', 'Centros Poblados y Rural Disperso', 'Total']
resultado_final['CLASE_AGRUPADA'] = pd.Categorical(
    resultado_final['CLASE_AGRUPADA'],
    categories=orden_categorias,
    ordered=True
)
resultado_final = resultado_final.sort_values(['CLASE_AGRUPADA']).reset_index(drop=True)

cols_numericas = resultado_final.select_dtypes(include=['float64', 'int64']).columns
resultado_final[cols_numericas] = resultado_final[cols_numericas].round(9)
resultado_final['P1_DEPARTAMENTO'] = 0
resultado_final

,CLASE_AGRUPADA,total,sin_deficit,deficit_cuanti,deficit_cuali,tipo_viv,mat_paredes,cohab,h_mitigable,P1_DEPARTAMENTO
0,Cabecera,1.132456e+07,9.722985e+06,388675.730015,1.212901e+06,299838.627496,299838.627496,23028.003899,23028.003899,0
1,Total,1.132456e+07,9.722985e+06,388675.730015,1.212901e+06,299838.627496,299838.627496,23028.003899,23028.003899,0


In [ ]:
fex = data['FEX_C_2018']
data_agg = data.copy()
if "P4000" in data_agg.columns:
    data_agg = data_agg[data_agg["P4000"] != 4]
if "P1070" in data_agg.columns:
    data_agg = data_agg[data_agg["P1070"] != 4]

data_agg['total'] = fex
data_agg['sin_deficit'] = (data_agg['CLASIFICACION'] == 'Sin deficit') * fex
data_agg['deficit_cuanti'] = (data_agg['CLASIFICACION'] == 'Deficit cuantitativo') * fex
data_agg['deficit_cuali'] = (data_agg['CLASIFICACION'] == 'Deficit cualitativo') * fex
data_agg['tipo_viv'] = (data_agg['IS_MAT_PAREDES'] == 1) * fex
data_agg['mat_paredes'] = (data_agg['IS_MAT_PAREDES'] == 1) * fex
data_agg['cohab'] = (data_agg['IS_COHAB'] == 1) * fex
data_agg['h_mitigable'] = (data_agg['IS_H_MITIG'] == 1) * fex

columnas_a_agrupar = [
    'total', 'sin_deficit', 'deficit_cuanti', 'deficit_cuali', 'tipo_viv', 'mat_paredes', 'cohab', 'h_mitigable'
]

mapeo_clase = {
    1: 'Cabecera',
    2: 'Centros Poblados y Rural Disperso',
    3: 'Centros Poblados y Rural Disperso'
}
data_agg['CLASE_AGRUPADA'] = data_agg['CLASE'].map(mapeo_clase)

resultado_detallado = data_agg.groupby(['P1_DEPARTAMENTO', 'CLASE_AGRUPADA'])[columnas_a_agrupar].sum().reset_index()

resultado_total = data_agg.groupby(['P1_DEPARTAMENTO'])[columnas_a_agrupar].sum().reset_index()
resultado_total['CLASE_AGRUPADA'] = 'Total'

resultado_final = pd.concat([resultado_detallado, resultado_total], ignore_index=True)

orden_categorias = ['Cabecera', 'Centros Poblados y Rural Disperso', 'Total']
resultado_final['CLASE_AGRUPADA'] = pd.Categorical(
    resultado_final['CLASE_AGRUPADA'],
    categories=orden_categorias,
    ordered=True
)
resultado_final = resultado_final.sort_values(['P1_DEPARTAMENTO', 'CLASE_AGRUPADA']).reset_index(drop=True)

cols_numericas = resultado_final.select_dtypes(include=['float64', 'int64']).columns
resultado_final[cols_numericas] = resultado_final[cols_numericas].round(9)

In [ ]:
nacional_por_clase = resultado_detallado.groupby('CLASE_AGRUPADA')[columnas_a_agrupar].sum().reset_index()
nacional_por_clase['P1_DEPARTAMENTO'] = 0

nacional_total_vals = nacional_por_clase[columnas_a_agrupar].sum()
nacional_total_row = pd.DataFrame([{
    'P1_DEPARTAMENTO': 0,
    'CLASE_AGRUPADA': 'Total',
    **nacional_total_vals.to_dict()
}])

nacional_agregado = pd.concat([nacional_por_clase, nacional_total_row], ignore_index=True)

resultado_final = pd.concat([resultado_final, nacional_agregado], ignore_index=True)

orden_categorias = ['Cabecera', 'Centros Poblados y Rural Disperso', 'Total']
resultado_final['CLASE_AGRUPADA'] = pd.Categorical(
    resultado_final['CLASE_AGRUPADA'],
    categories=orden_categorias,
    ordered=True
)

resultado_final = resultado_final.sort_values(['P1_DEPARTAMENTO', 'CLASE_AGRUPADA']).reset_index(drop=True)

cols_numericas = resultado_final.select_dtypes(include=['float64', 'int64']).columns
resultado_final[cols_numericas] = resultado_final[cols_numericas].round(9)

In [ ]:
resultado_final

,CLASE_AGRUPADA,total,sin_deficit,deficit_cuanti,deficit_cuali,tipo_viv,mat_paredes,cohab,h_mitigable,P1_DEPARTAMENTO
0,Cabecera,1.132456e+07,9.722985e+06,388675.730015,1.212901e+06,299838.627496,299838.627496,23028.003899,23028.003899,0
1,Total,1.132456e+07,9.722985e+06,388675.730015,1.212901e+06,299838.627496,299838.627496,23028.003899,23028.003899,0


In [ ]:
#proyecciones_df = pd.read_csv('proyeccion_vivienda.csv', delimiter="\t")

In [ ]:
#proyecciones_df

In [ ]:
'''
proyecciones_2016 = proyecciones_df[['Dpto', 'Clase', '2016']].copy()
proyecciones_2016 = proyecciones_2016.rename(columns={'2016': 'POBLACION_2016'})

proyecciones_2016['Dpto'] = proyecciones_2016['Dpto'].astype(int)

resultado_merged = resultado_final.merge(
    proyecciones_2016,
    on=['Dpto', 'Clase'],
    how='left'
)
'''
resultado_final['Dpto'] = resultado_final['P1_DEPARTAMENTO'].fillna(0).astype(int)
resultado_final['Clase'] = resultado_final['CLASE_AGRUPADA'].astype(str)

for col in columnas_a_agrupar:
    resultado_final[f'{col}_PROP'] = resultado_final[col] / resultado_final['total'] * 100

resultado_final_prop = resultado_final.copy()

cols_prop = ['sin_deficit_PROP', 'deficit_cuanti_PROP', 'deficit_cuali_PROP', 'tipo_viv_PROP', 'mat_paredes_PROP', 'cohab_PROP', 'h_mitigable_PROP']

resultado_final_prop[cols_prop] = resultado_final_prop[cols_prop].round(9)

cols = ['Dpto','Clase'] + cols_prop
resultado_final_prop = resultado_final_prop[cols]
resultado_final_prop.insert(0, 'Año', year)
resultado_final_prop.to_csv(f"df_prop_{year}.csv")

In [ ]:
resultado_final_prop

,Año,Dpto,Clase,sin_deficit_PROP,deficit_cuanti_PROP,deficit_cuali_PROP,tipo_viv_PROP,mat_paredes_PROP,cohab_PROP,h_mitigable_PROP
0,2017,0,Cabecera,85.857499,3.432148,10.710353,2.647684,2.647684,0.203346,0.203346
1,2017,0,Total,85.857499,3.432148,10.710353,2.647684,2.647684,0.203346,0.203346


In [ ]:

data_full = pd.concat(
    [pd.read_csv(f"df_prop_{anio}.csv") for anio in range(2010, 2018)],
    ignore_index=True
)
data_full['Dpto'] = data_full['Dpto'].fillna(0).astype(int)
data_full['Año'] = data_full['Año'].fillna(0).astype(int)
del data_full['Unnamed: 0']
data_full.to_csv('df_prop_2010_2017.csv', index=False)

